In [117]:
%pip install lightgbm


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Section1. 환자의 당뇨병 여부 예측

- **평가**: ROC-AUC
- **target**: Outcome (0:정상, 1:당뇨병) 

In [118]:
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/diabetes_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/diabetes_test.csv")


print(train.shape, test.shape)
# print(train.head())
# print(train.info())

target = train.pop('Outcome')
print(train.shape, test.shape)

from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

models = {
    "RF": RandomForestClassifier(random_state=42),
    "RF2": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "LGBM": LGBMClassifier(random_state=42, verbose=-1),
    "LGBM2": LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict_proba(X_val)
    score = roc_auc_score(y_val, pred[:,1])
    scores[name] = score
    print(name, score)

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict_proba(test)

submit = pd.DataFrame({'pred':pred[:,1]})
submit.to_csv("data/section1.csv", index=False)

print(pd.read_csv("data/section1.csv").head())
print(pd.read_csv("data/section1.csv").shape)

(614, 9) (154, 8)
(614, 8) (154, 8)
(491, 8) (123, 8) (491,) (123,)
RF 0.8257032542746828
RF2 0.8306674020959736
LGBM 0.8063982349696635
LGBM2 0.8033645890788748
RF2 RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
       pred
0  0.242176
1  0.265287
2  0.174212
3  0.061439
4  0.062724
(154, 1)


# Section 2. 신용 등급 예측

- **평가**: f1-macro
- **target**: Credit_Score (Good, Standard, Poor)

In [119]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/score_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/score_test.csv")

print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())
# print(test.isnull().sum())
# print(train['Credit_Score'].value_counts())

target = train.pop('Credit_Score')
print(train.shape, test.shape)

num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(include='object').columns
# print(len(cat_cols))

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)

    if same:
        print(col, "\t 같음")
    else:
        print(col)


# train = pd.get_dummies(train)
# test = pd.get_dummies(test)
# print(train.shape, test.shape) # (4198, 29) (1499, 29)

n = len(train)
data = pd.concat([train, test], axis=0)
data_oh = pd.get_dummies(data)

train = data_oh.iloc[:n].copy()
test = data_oh.iloc[n:].copy()
print(train.shape, test.shape)


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42, stratify=target)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

models = {
    "RF": RandomForestClassifier(random_state=42),
    "LGBM": LGBMClassifier(random_state=42, verbose=-1)
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    score = f1_score(y_val, pred, average='macro')
    scores[name] = score
    print(name, score)

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict(test)

submit = pd.DataFrame({'pred':pred})
submit.to_csv("data/section2.csv", index=False)

print(pd.read_csv("data/section2.csv").head())
print(pd.read_csv("data/section2.csv").shape)

(4198, 21) (1499, 20)
(4198, 20) (1499, 20)
Payment_of_Min_Amount 	 같음
Credit_Mix 	 같음
Payment_Behaviour 	 같음
(4198, 29) (1499, 29)
(3358, 29) (840, 29) (3358,) (840,)


/var/folders/9h/qj2rv5k9005fgsp61dxxrk100000gn/T/ipykernel_33987/1496537955.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include='object').columns


RF 0.7088934821088482
LGBM 0.6953271267531284
RF RandomForestClassifier(random_state=42)
       pred
0      Poor
1      Good
2  Standard
3      Good
4  Standard
(1499, 1)


# Section 3. 항공권 가격 예측

- **평가**: RMSE
- **target**: price

In [120]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/flight_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/flight_test.csv")


print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())
# print(test.isnull().sum())

train = train.drop(['flight'], axis=1)
test = test.drop(['flight'], axis=1)
print(train.shape, test.shape)

target = train.pop('price')
print(train.shape, test.shape)

num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(exclude='number').columns

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)
    # if same:
    #     print(col, " 같음")
    # else:
    #     print(col, " 다름")


n = len(train)

data = pd.concat([train, test])
data_oh = pd.get_dummies(data)

train = data_oh.iloc[:n].copy()
test = data_oh.iloc[n:].copy()
print(train.shape, test.shape)


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

models = {
    "RF": RandomForestRegressor(random_state=42),
    "LGBM": LGBMRegressor(random_state=42, verbose=-1)
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    score = root_mean_squared_error(y_val, pred)
    scores[name] = score
    print(name, score)

best_name = min(scores, key=scores.get)
best_model = models[best_name]

best_model.fit(train, target)
pred = best_model.predict(test)

submit = pd.DataFrame({'pred':pred})
submit.to_csv("data/section3.csv", index=False)

print(pd.read_csv("data/section3.csv").head())
print(pd.read_csv("data/section3.csv").shape)

(10505, 11) (4502, 10)
(10505, 10) (4502, 9)
(10505, 9) (4502, 9)
(10505, 37) (4502, 37)
(8404, 37) (2101, 37) (8404,) (2101,)
RF 4341.463479340969
LGBM 4424.495915794654
       pred
0  56152.32
1   5452.14
2  12371.16
3   5959.52
4   5115.00
(4502, 1)


# Section 4. 이직 여부 예측

- **평가**: ROC-AUC
- **target**: target

In [121]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/hr_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/hr_test.csv")

print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())
# print(test.isnull().sum())
# print(train['city'].value_counts())

target = train.pop('target')
print(train.shape, test.shape)

train = train.fillna("X")
test = test.fillna("X")
# print(train.isnull().sum())

# train = train.drop(['city'], axis=1)
# test = test.drop(['city'], axis=1)
# print(train.shape, test.shape)

num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(exclude='number').columns

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)
    # if same:
    #     print(col)
    # else:
    #     print(col, "다름르르르르를")


n = len(train)
data = pd.concat([train, test])
data_oh = pd.get_dummies(data)
train = data_oh.iloc[:n].copy()
test = data_oh.iloc[n:].copy()
# print(train.shape, test.shape)


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

models = {
    "RF": RandomForestClassifier(random_state=42),
    "LGBM": LGBMClassifier(random_state=42, verbose=-1)
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict_proba(X_val)
    score = roc_auc_score(y_val, pred[:,1])
    scores[name] = score
    print(name, score)

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict_proba(test)

submit = pd.DataFrame({'pred':pred[:,1]})
submit.to_csv("data/section4.csv", index=False)

print(pd.read_csv("data/section4.csv").head())
print(pd.read_csv("data/section4.csv").shape)

(15326, 14) (3832, 13)
(15326, 13) (3832, 13)
(12260, 195) (3066, 195) (12260,) (3066,)
RF 0.7966271302523777
LGBM 0.8043087909537169
LGBM LGBMClassifier(random_state=42, verbose=-1)
       pred
0  0.215958
1  0.504452
2  0.584754
3  0.044629
4  0.095243
(3832, 1)


# Section 5. 약물 종류 예측

- **평가**: f1 macro
- **target**: Drug

In [122]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/drug_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/drug_test.csv")

print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())
# print(test.isnull().sum())

target = train.pop('Drug')
print(train.shape, test.shape)

num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(exclude='number').columns

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)
    # if same:
    #     print(col)
    # else:
    #     print(col, " 다르르르으으으음")


# train = pd.get_dummies(train)
# test = pd.get_dummies(test)
# print(train.shape, test.shape)
# (80, 9) (100, 9)

n = len(train)
data = pd.concat([train, test])
data_oh = pd.get_dummies(data)
train = data_oh.iloc[:n].copy()
test = data_oh.iloc[n:].copy()
print(train.shape, test.shape)


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42, stratify=target)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

models = {
    "RF": RandomForestClassifier(random_state=0),
    "LGBM": LGBMClassifier(random_state=0, verbose=-1)
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    score = f1_score(y_val, pred, average='macro')
    scores[name] = score
    print(name, score)

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict(test)

submit = pd.DataFrame({'pred':pred})
submit.to_csv("data/section5.csv", index=False)

print(pd.read_csv("data/section5.csv").head())
print(pd.read_csv("data/section5.csv").shape)



(80, 6) (100, 5)
(80, 5) (100, 5)
(80, 9) (100, 9)
(64, 9) (16, 9) (64,) (16,)
RF 0.7818181818181819
LGBM 0.6751515151515152
RF RandomForestClassifier(random_state=0)
    pred
0  DrugY
1  DrugY
2  DrugY
3  DrugY
4  drugA
(100, 1)


# Section 6. 노트북 가격 예측

- **평가**: R2
- **target**: Price

In [123]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/laptop_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/laptop_test.csv")

print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())
# print(test.isnull().sum().sum())

target = train.pop('Price')
print(train.shape, test.shape)

num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(exclude='number').columns

train['RAM'] = train['RAM'].fillna(-1)
train[cat_cols] = train[cat_cols].fillna('X')

test['RAM'] = test['RAM'].fillna(-1)
test[cat_cols] = test[cat_cols].fillna('X')

# print(train.isnull().sum().sum())
# print(test.isnull().sum().sum())

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)
    # if same:
    #     print(col)
    # else:
    #     print(col, "다르르르름")


n = len(train)
data = pd.concat([train, test])
data_oh = pd.get_dummies(data)

train = data_oh.iloc[:n].copy()
test = data_oh.iloc[n:].copy()
print(train.shape, test.shape)


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

models = {
    "RF": RandomForestRegressor(random_state=0),
    "LGBM": LGBMRegressor(random_state=0, verbose=-1),
    "LGBM2": LGBMRegressor(
        random_state=0,
        min_child_samples=5,
        verbose=-1
    )
}

scores = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    score = r2_score(y_val, pred)
    scores[name] = score
    print(name, score)

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict(test)

submit = pd.DataFrame({'pred': pred})
submit.to_csv("data/section6.csv", index=False)

print(pd.read_csv("data/section6.csv").head())
print(pd.read_csv("data/section6.csv").shape)

(91, 10) (39, 9)
(91, 9) (39, 9)
(91, 119) (39, 119)
(72, 119) (19, 119) (72,) (19,)
RF 0.7973068266810601
LGBM 0.4818775493904527
LGBM2 0.6407996181833291
RF RandomForestRegressor(random_state=0)
           pred
0  41499.460000
1  40473.972035
2  59915.860000
3  40218.489177
4  46446.800000
(39, 1)
